## Knižnice

In [ ]:
# 1. Základná manipulácia s dátami a numerika
import pandas as pd
import numpy as np
import os
import wandb
from kaggle_secrets import UserSecretsClient

# 2. Vizualizácia dát
import matplotlib.pyplot as plt  
import seaborn as sns  
import tensorflow as tf

from collections import defaultdict
import cv2

## Pozrieme sa, v akom tvare su nase data

In [ ]:
DATA_DIR = '/kaggle/input/american-sign-language/ASL_Dataset'
# Pozrime sa, co je vnutri
print(os.listdir(DATA_DIR))  

In [ ]:
TRAIN_DIR = os.path.join(DATA_DIR, 'Train')
TEST_DIR = os.path.join(DATA_DIR, 'Test')

# Pozrime sa, ake su tam triedy (pismena)
CLASSES = os.listdir(TRAIN_DIR)
CLASSES.sort()

print(f"Našiel som tieto triedy: {CLASSES}\n")

print(f"Celkovo máme {len(CLASSES)} rôznych znakov")

## Načítanie dát

In [ ]:
IMG_SIZE = (128,128) # 128, lebo 224 je veľa
BATCH_SIZE = 32

total_train_images = sum(len(files) for _, _, files in os.walk(TRAIN_DIR))
total_test_images  = sum(len(files) for _, _, files in os.walk(TEST_DIR))

print("Počty obrázkov načítané zo systému:")
print(f"  • Tréningový priečinok (pred splitom): {total_train_images}")
print(f"  • Testovací priečinok: {total_test_images}")

# --- Dataset SPLIT ---
train_data = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='grayscale',
    label_mode='categorical',
    shuffle=True,
    validation_split=0.2,
    subset="training",
    seed=42,
)

val_data = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb',
    label_mode='categorical',
    shuffle=True,
    validation_split=0.2,
    subset="validation",
    seed=42
)

test_data = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='grayscale',
    label_mode='categorical',
    shuffle=False
)

# --- Presné počty v jednotlivých datasetoch ---
train_count = 0
for batch, _ in train_data:
    train_count += batch.shape[0]

val_count = 0
for batch, _ in val_data:
    val_count += batch.shape[0]

test_count = 0
for batch, _ in test_data:
    test_count += batch.shape[0]

# --- Výpis ---
print("\nRozdelenie datasetu:")
print(f"  • Tréningová množina:  {train_count} obrázkov  ({train_count / total_train_images * 100:.2f} %)")
print(f"  • Validačná množina:   {val_count} obrázkov ({val_count / total_train_images * 100:.2f} %)")
print(f"  • Testovacia množina:  {test_count} obrázkov")


In [ ]:
import hashlib


def hash_file(path):
    hasher = hashlib.sha1()
    with open(path, "rb") as f:
        buf = f.read()
        hasher.update(buf)
    return hasher.hexdigest()

In [ ]:
def najdi_duplicitne_obrazky(root_dir):
    hash_map = defaultdict(list)

    for trieda in os.listdir(root_dir):
        cesta_triedy = os.path.join(root_dir, trieda)
        if not os.path.isdir(cesta_triedy):
            continue

        for subor in os.listdir(cesta_triedy):
            cesta_obr = os.path.join(cesta_triedy, subor)
            try:
                h = hash_file(cesta_obr)
                hash_map[h].append(cesta_obr)
            except:
                print(f"Nepodarilo sa prečítať: {cesta_obr}")

    duplicates = {h: paths for h, paths in hash_map.items() if len(paths) > 1}
    return duplicates

In [ ]:
duplicates = najdi_duplicitne_obrazky(TRAIN_DIR)
if len(duplicates) == 0:
    print("Žiadne duplicitné obrázky neboli nájdené!")
else:
    for h, paths in duplicates.items():
        print(f"Duplicate hash: {h}")
        for p in paths:
            print("  -", p)
        print()

## EDA

1. Zobrazit kolko mame dat
2. Zobrazit ako vyzeraju
3. Zobrazit ake su velke
4. Zobrazit aky maju pocet kanalov

In [ ]:
counts = {}
for pismeno in CLASSES:
    path = os.path.join(TRAIN_DIR, pismeno)
    counts[pismeno] = len(os.listdir(path))

# Vykreslenie
plt.figure(figsize=(12, 6))
plt.bar(counts.keys(), counts.values())
plt.title("Počty obrázkov v trénovacích triedach")
plt.xlabel("Trieda (písmeno)")
plt.ylabel("Počet obrázkov")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

### Môžeme vidieť rozdiel v množstve dát pre písmeno U oproti ostatným znakom. Neskôr to budeme pravdepodobne riesiť UNDERSAMPLINGOM.

Zobrazíme si pár obrazkov nech vieme, s čím pracujeme.

In [ ]:
# Zoberieme si jednu varku (batch) z trenovacich dat
for images, labels in train_data.take(1):
    # images = varka 32 obrazkov (lebo BATCH_SIZE=32)
    # labels = varka 32 labelov

    plt.figure(figsize=(10, 10))
    for i in range(9): # Zobrazime prvych 9 obrazkov z varky
        ax = plt.subplot(3, 3, i + 1)

        # Obrazky su normalizovane (0-255), pre plt.imshow ich musime pretypovat
        plt.imshow(images[i].numpy().astype("uint8"), cmap='gray')

        # Labely su 'categorical', takze musime najst index s najvyssou hodnotou
        label_index = np.argmax(labels[i])
        plt.title(f"Label: {train_data.class_names[label_index]}")
        plt.axis("off")

    plt.show()

Pozrieme sa na to, akú majú obrázky veľkosť

In [ ]:
from PIL import Image

def ziskaj_rozmery_obrazkov_fast(root_dir):
    sirky, vysky, kanaly = [], [], []

    for trieda in os.listdir(root_dir):
        cesta_triedy = os.path.join(root_dir, trieda)
        if not os.path.isdir(cesta_triedy):
            continue

        for subor in os.listdir(cesta_triedy):
            cesta_obr = os.path.join(cesta_triedy, subor)

            try:
                with Image.open(cesta_obr) as img:
                    w, h = img.size
                    c = len(img.getbands())   # napr. ("R","G","B") → 3

                    sirky.append(w)
                    vysky.append(h)
                    kanaly.append(c)

            except Exception as e:
                print("Chyba pri načítaní:", cesta_obr)

    return sirky, vysky, kanaly


In [ ]:
sirky, vysky, kanaly = ziskaj_rozmery_obrazkov_fast(TRAIN_DIR)

print("Celkový počet načítaných obrázkov:", len(sirky))
print("Min šírka:", min(sirky), " | Max šírka:", max(sirky))
print("Min výška:", min(vysky), " | Max výška:", max(vysky))
print("Kanály (unikátne hodnoty):", set(kanaly))

plt.figure(figsize=(18, 5))

# Šírky
plt.subplot(1, 3, 1)
plt.hist(sirky, bins=20)
plt.title("Histogram šírok obrázkov")
plt.xlabel("Šírka (px)")
plt.ylabel("Počet obrázkov")
plt.grid(True, linestyle="--", alpha=0.4)

# Výšky
plt.subplot(1, 3, 2)
plt.hist(vysky, bins=20)
plt.title("Histogram výšok obrázkov")
plt.xlabel("Výška (px)")
plt.ylabel("Počet obrázkov")
plt.grid(True, linestyle="--", alpha=0.4)

# Kanály
plt.subplot(1, 3, 3)
plt.hist(kanaly, bins=5)
plt.title("Histogram počtu kanálov")
plt.xlabel("Počet kanálov")
plt.ylabel("Počet obrázkov")
plt.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

Vidime ze obrazky su farebné (pocet kanalov 3 = RGB ) - neskôr bude vhodné ich prerobiť do greyscale pre menšie vyťaženie modelu.

wandb

## Preprocessing dát

V tejto sekcii aplikujeme **normalizáciu** a **data augmentation** na naše dáta.


### 1. Normalizácia dát

Normalizujeme obrázky z rozsahu [0, 255] na [0, 1] a podľa vypočítaného meanu a std, pre lepšiu konvergenciu tréningu.


### 2. Data Augmentation

Data augmentation pomáha zlepšiť generalizáciu modelu tým, že vytvára variácie trénovacích dát. Použijeme:
- Náhodné rotácie
- Náhodné posuny (translation)
- Náhodné preklopenie (horizontal flip)
- Náhodný zoom
- Náhodné zmeny jasu a kontrastu


In [ ]:
'''
def get_stats_from_raw(dataset, num_batches=10000):
    sum_img = 0.0
    sum_sq_img = 0.0
    count = 0.0

    for images, _ in dataset.take(num_batches):
        images = tf.cast(images, tf.float32) / 255.0

        sum_img += tf.reduce_sum(images)
        sum_sq_img += tf.reduce_sum(tf.square(images))
        count += tf.cast(tf.size(images), tf.float32)
    mean = sum_img / count
    std = tf.sqrt((sum_sq_img / count) - tf.square(mean))

    return mean.numpy(), std.numpy()


CALCULATED_MEAN, CALCULATED_STD = get_stats_from_raw(train_data)
print(f"MEAN: {CALCULATED_MEAN:.4f}")
print(f"STD:  {CALCULATED_STD:.4f}")
'''

In [ ]:
from tensorflow.keras import layers
STD=0.1458
MEAN=0.5355



data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(factor=0.03, fill_mode='constant'),
    layers.RandomTranslation(height_factor=0.005, width_factor=0.005, fill_mode='constant'),
    layers.RandomZoom(height_factor=0.005, width_factor=0.005, fill_mode='constant'),
    layers.RandomBrightness(factor=0.002),
    layers.RandomContrast(factor=0.002),
], name='data_augmentation')

def preprocess_train(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = data_augmentation(image, training=True)
    safe_std = 1.0 if STD == 0 else STD
    image = (image - MEAN) / safe_std
    return image, label

def preprocess_val(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    safe_std = 1.0 if STD == 0 else STD
    image = (image - MEAN) / safe_std
    return image, label


#Trénovacie dáta
train_ds_final = train_data.map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
train_ds_final = train_ds_final.shuffle(buffer_size=1000).prefetch(tf.data.AUTOTUNE)

# Validačné dáta
test_ds_final = test_data.map(preprocess_val, num_parallel_calls=tf.data.AUTOTUNE)
test_ds_final = test_ds_final.prefetch(tf.data.AUTOTUNE)

### 3. Data standardization

### 3. Vizualizácia normalizovaných dát

Pozrime sa, ako vyzerajú augmentované obrázky:


In [ ]:
import matplotlib.pyplot as plt

def denormalize_for_view(image):
    # Vzorec: x_povodna = (x_norm * std) + mean
    safe_std = 1.0 if STD == 0 else STD
    image = (image * safe_std) + MEAN
    return tf.clip_by_value(image, 0.0, 1.0)
images, labels = next(iter(train_ds_final.take(1)))


plt.figure(figsize=(15, 5))
for i in range(5):
    ax = plt.subplot(1, 5, i + 1)
    img_to_show = denormalize_for_view(images[i])
    plt.imshow(img_to_show.numpy(), cmap='gray')
    if len(labels[i].shape) > 0 and labels[i].shape[0] > 1:
        label_val = np.argmax(labels[i])
    else:
        label_val = int(labels[i])

    try:
        title = train_data.class_names[label_val]
    except:
        title = str(label_val)
    plt.title(f"Label: {title}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
wandb_key = UserSecretsClient().get_secret("wandb_api")
wandb.login(key=wandb_key)

#### Vlastna modularna cnn, s ktorou budeme experimentovat

In [ ]:
# Jednoduchá modulárna CNN architektúra pre ASL klasifikáciu
import tensorflow as tf
from typing import List, Optional, Tuple
from tensorflow.keras import layers, models

class SimpleCNN:
    
    def __init__(
        self,
        input_shape: Tuple[int, int, int] = (128,128, 1),
        num_classes: int = 28,
        num_conv_layers: int = 3,
        filters_per_layer: Optional[List[int]] = None,
        dense_units: Optional[List[int]] = None,
        dropout_rate: float = 0.5,
        use_batch_norm: bool = True,
        use_pooling: bool = True
    ):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.num_conv_layers = num_conv_layers
        self.dropout_rate = dropout_rate
        self.use_batch_norm = use_batch_norm
        self.use_pooling = use_pooling
        
        # Default filtre
        if filters_per_layer is None:
            self.filters = [32 * (2 ** i) for i in range(num_conv_layers)]
        elif isinstance(filters_per_layer, int):
            self.filters = [filters_per_layer] * num_conv_layers
        else:
            if len(filters_per_layer) != num_conv_layers:
                raise ValueError(f"Počet filtrov ({len(filters_per_layer)}) sa musí rovnať počtu vrstiev ({num_conv_layers})")
            self.filters = filters_per_layer
        
        # Default dense units
        if dense_units is None:
            self.dense_units = [256, 128]
        else:
            self.dense_units = dense_units
    
    def build(self):
        """Vytvorí a vráti model."""
        inputs = layers.Input(shape=self.input_shape, name='input')
        x = inputs
        
        # Konvolučné vrstvy
        for i in range(self.num_conv_layers):
            x = layers.Conv2D(
                filters=self.filters[i],
                kernel_size=3,
                padding='same',
                activation='relu',
                name=f'conv_{i+1}'
            )(x)
            
            if self.use_batch_norm:
                x = layers.BatchNormalization(name=f'bn_{i+1}')(x)
            
            if self.use_pooling:
                x = layers.MaxPooling2D(pool_size=2, name=f'pool_{i+1}')(x)
        
        # Global Average Pooling
        x = layers.GlobalAveragePooling2D(name='global_pool')(x)
        
        # Dense vrstvy
        for i, units in enumerate(self.dense_units):
            x = layers.Dense(units, activation='relu', name=f'dense_{i+1}')(x)
            x = layers.Dropout(self.dropout_rate, name=f'dropout_{i+1}')(x)
        
        # Výstupná vrstva
        outputs = layers.Dense(
            self.num_classes,
            activation='softmax',
            name='output'
        )(x)
        
        model = models.Model(inputs=inputs, outputs=outputs, name='SimpleCNN')
        return model
    
    def compile(
        self,
        optimizer: str = 'adam',
        learning_rate: float = 0.001,
        loss: str = 'categorical_crossentropy',
        metrics: Optional[List[str]] = None
    ):
        """Vytvorí, skompiluje a vráti model."""
        model = self.build()
        
        if optimizer.lower() == 'adam':
            opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)
        elif optimizer.lower() == 'sgd':
            opt = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
        else:
            opt = tf.keras.optimizers.get(optimizer)
            if hasattr(opt, 'learning_rate'):
                opt.learning_rate = learning_rate
        
        if metrics is None:
            metrics = ['accuracy']
        
        model.compile(optimizer=opt, loss=loss, metrics=metrics)
        return model


### Tu experimentujeme CNN

In [ ]:

NUM_CLASSES = 28

cnn_model = SimpleCNN(
    input_shape=(*IMG_SIZE, 1),
    num_classes=NUM_CLASSES,
    num_conv_layers=3,  # Môžete zmeniť počet vrstiev
    filters_per_layer=[32, 64, 128],  # Môžete zmeniť počet filtrov
    dense_units=[128],  # Môžete zmeniť počet dense vrstiev
    dropout_rate=0.5,
    use_batch_norm=True,
    use_pooling=True
)

cnn_model = cnn_model.compile(
    optimizer='adam',
    learning_rate=0.001,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks_cnn = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        'best_cnn_model.keras',
        save_best_only=True,
        monitor='val_loss',
        verbose=1
    )
]

print("\nZačínam tréning CNN modelu...")
history_cnn = cnn_model.fit(
    train_ds_final,
    epochs=10,
    validation_data=test_ds_final,
    callbacks=callbacks_cnn,
    verbose=1
)

test_loss, test_accuracy = cnn_model.evaluate(test_ds_final, verbose=1)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


# ResNet-50

### Prvé sme sa rozhodli urobiť Res-Net50. Prečo?? Prečo nie, pýtaš sa príliš veľa otázok...

(páči sa mi že má skip connections, je pomerne rýchly a fajn pre stredne veľké množstvo dát, ako máme my)

In [ ]:
# Preprocessing už bol vykonaný vyššie v sekcii "Preprocessing dát"
# Používame premenné: train_ds_augmented (pre tréning) a test_ds (pre testovanie)
# Tieto premenné obsahujú normalizované dáta s augmentáciou (pre tréning) alebo bez (pre testovanie)

In [ ]:
'''
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models

NUM_CLASSES = 28

# 1. Stiahnutie predtrénovaného ResNet50 (bez hornej klasifikačnej vrstvy)
base_model = ResNet50(
    weights='imagenet',  # Použijeme váhy natrénované na ImageNet
    include_top=False,   # Odstránime poslednú vrstvu
    input_shape=(224, 224, 3)
)

# 2. Zmrazenie váh base modelu
base_model.trainable = False

# 3. Vytvorenie nového modelu
model = models.Sequential([
    # Augmentácia dát (náhodné preklopenie/otočenie pre lepšiu robustnosť)
    # layers.RandomFlip("horizontal"),
    # layers.RandomRotation(0.1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(256, activation='relu'),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

# 4. Kompilácia modelu
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
'''

In [ ]:
'''
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import wandb


from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint
wandb.init(project="ASL_ResNet", config={"model": "ResNet50", "epochs": 10})

callbacks = [
    EarlyStopping(
        monitor='val_loss', 
        patience=3, 
        restore_best_weights=True,
        verbose=1
    ),
    
    ModelCheckpoint(
        "best_asl_resnet.keras", 
        save_best_only=True, 
        monitor='val_loss',
        verbose=1
    ),
    WandbMetricsLogger()
]


history = model.fit(
    train_ds_augmented,  # Používame augmentované trénovacie dáta z preprocessing sekcie
    epochs=10,
    validation_data=test_ds,  # Testovacie dáta (normalizované, bez augmentácie)
    callbacks=callbacks
)
wandb.finish()
'''

Do budúcnosti premýľame že výskúšame aj tieto modeli

EfficientNet, MobileNetV3